In [ ]:
# 02. CHART Data Cleaning and Feature Engineering

This notebook transforms raw MDOT CHART event logs into a structured dataset suitable for modeling.

## Objectives

- Standardize multi-year CHART data
- Parse time and duration fields
- Extract route and direction from location text
- Create a modeling-ready incident dataset

## Outputs

- `chart_events_clean.parquet`
- `chart_model_incident.parquet`

In [ ]:
## 1. Configuration

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import warnings

warnings.filterwarnings("ignore")

# =========================
# PATHS
# =========================
RAW_BASE = Path(r"C:\")
PROJECT_ROOT = Path(r"\\")

CHART_DIR = RAW_BASE / "MDOT_CHART"

OUTPUT_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_DATA_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("CHART_DIR exists:", CHART_DIR.exists())
print("OUTPUT_TABLE_DIR:", OUTPUT_TABLE_DIR)
print("OUTPUT_DATA_DIR:", OUTPUT_DATA_DIR)

CHART_DIR exists: True
OUTPUT_TABLE_DIR: \tables
OUTPUT_DATA_DIR: \\traffic_Incident\data\processed


In [2]:
def list_csv_files(folder: Path):
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*.csv") if p.is_file()])


chart_files = list_csv_files(CHART_DIR)

print(f"CHART files found: {len(chart_files):,}")
for f in chart_files[:10]:
    print("-", f.name)

CHART files found: 5
- MDOT_CHART_2020.csv
- MDOT_CHART_2021.csv
- MDOT_CHART_2022.csv
- MDOT_CHART_2023.csv
- MDOT_CHART_2024.csv


In [ ]:
## 2. Helper Functions

In [3]:
def read_csv_safe(path, nrows=None, usecols=None):
    try:
        return pd.read_csv(path, nrows=nrows, usecols=usecols, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, nrows=nrows, usecols=usecols, encoding="latin1", low_memory=False)


def existing_col(columns, candidates):
    mapping = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        key = cand.strip().lower()
        if key in mapping:
            return mapping[key]
    return None


def infer_year(path: Path):
    m = re.search(r"(20\d{2})", str(path))
    return int(m.group(1)) if m else np.nan


def clean_string_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .str.strip()
         .replace({"<NA>": pd.NA, "nan": pd.NA, "None": pd.NA})
    )


def extract_route(location: pd.Series) -> pd.Series:
    """
    location text에서 route 추출
    예: I-95, I 95, MD 295, US 50, US-1, SR 32 등
    """
    pat = r"\b(I[-\s]?\d+|US[-\s]?\d+|MD[-\s]?\d+|SR[-\s]?\d+|RT[-\s]?\d+)\b"
    out = location.str.extract(pat, expand=False)
    if out is None:
        return pd.Series(pd.NA, index=location.index, dtype="string")
    return out.astype("string").str.upper().str.replace(r"\s+", "-", regex=True)


def extract_direction(location: pd.Series) -> pd.Series:
    pat = r"\b(NORTH|SOUTH|EAST|WEST|NORTHBOUND|SOUTHBOUND|EASTBOUND|WESTBOUND|INNER LOOP|OUTER LOOP|NB|SB|EB|WB)\b"
    out = location.str.extract(pat, expand=False)
    return out.astype("string").str.upper()


def extract_exit(location: pd.Series) -> pd.Series:
    pat = r"\bEXIT\s+([0-9A-Z]+)\b"
    out = location.str.extract(pat, expand=False)
    return out.astype("string").str.upper()


def extract_milepost(location: pd.Series) -> pd.Series:
    pat = r"\b(?:MP|MM|MILEPOST)\s*([0-9]+(?:\.[0-9]+)?)\b"
    out = location.str.extract(pat, expand=False)
    return pd.to_numeric(out, errors="coerce")


def parse_chart_file(path: Path) -> tuple[pd.DataFrame, dict]:
    """
    CHART raw csv -> cleaned dataframe + quality summary row
    """
    df = read_csv_safe(path)

    cols = list(df.columns)

    start_col = existing_col(cols, ["Start time", "start_time"])
    closed_col = existing_col(cols, ["Closed time", "closed_time"])
    type_col = existing_col(cols, ["Standardized Type", "standardized_type"])
    agency_type_col = existing_col(cols, ["Agency Specific Type", "agency_specific_type", "Agency-specific Type"])
    location_col = existing_col(cols, ["Location", "location"])
    responders_col = existing_col(cols, ["Responders", "responders"])
    lanes_col = existing_col(cols, ["Max Lanes Closed", "max_lanes_closed", "Max lanes closed"])
    op_center_col = existing_col(cols, ["Op. Center", "Op Center", "op_center"])
    event_id_col = existing_col(cols, ["Event ID", "event_id", "Id", "ID"])

    # time parsing
    start = pd.to_datetime(df[start_col], errors="coerce") if start_col else pd.Series(pd.NaT, index=df.index)
    closed = pd.to_datetime(df[closed_col], errors="coerce") if closed_col else pd.Series(pd.NaT, index=df.index)

    # 핵심 수정: .dt.total_seconds() 대신 Timedelta 나누기
    duration = closed - start
    duration_min = duration / pd.Timedelta(minutes=1)

    cleaned = pd.DataFrame({
        "source_file": path.name,
        "year_inferred": infer_year(path),
        "event_id": df[event_id_col].astype("string") if event_id_col else pd.Series(pd.NA, index=df.index, dtype="string"),
        "start_time": start,
        "closed_time": closed,
        "duration_min": duration_min,
        "event_type": clean_string_series(df[type_col]) if type_col else pd.Series(pd.NA, index=df.index, dtype="string"),
        "agency_type": clean_string_series(df[agency_type_col]) if agency_type_col else pd.Series(pd.NA, index=df.index, dtype="string"),
        "location_raw": clean_string_series(df[location_col]) if location_col else pd.Series(pd.NA, index=df.index, dtype="string"),
        "op_center": clean_string_series(df[op_center_col]) if op_center_col else pd.Series(pd.NA, index=df.index, dtype="string"),
        "responders": pd.to_numeric(df[responders_col], errors="coerce") if responders_col else pd.Series(np.nan, index=df.index),
        "max_lanes_closed": pd.to_numeric(df[lanes_col], errors="coerce") if lanes_col else pd.Series(np.nan, index=df.index),
    })

    # time features
    cleaned["year"] = cleaned["start_time"].dt.year
    cleaned["month"] = cleaned["start_time"].dt.month
    cleaned["day"] = cleaned["start_time"].dt.day
    cleaned["hour"] = cleaned["start_time"].dt.hour
    cleaned["day_of_week"] = cleaned["start_time"].dt.dayofweek
    cleaned["day_name"] = cleaned["start_time"].dt.day_name()
    cleaned["is_weekend"] = cleaned["day_of_week"].isin([5, 6]).astype("Int64")
    cleaned["is_peak_am"] = cleaned["hour"].between(6, 9, inclusive="both").astype("Int64")
    cleaned["is_peak_pm"] = cleaned["hour"].between(15, 18, inclusive="both").astype("Int64")

    # location parsing
    loc = cleaned["location_raw"].fillna("").astype("string").str.upper()
    cleaned["route"] = extract_route(loc)
    cleaned["direction"] = extract_direction(loc)
    cleaned["exit"] = extract_exit(loc)
    cleaned["milepost"] = extract_milepost(loc)

    # quality flags
    cleaned["duration_missing"] = cleaned["duration_min"].isna().astype("Int64")
    cleaned["duration_nonpositive"] = (cleaned["duration_min"] <= 0).astype("Int64")
    cleaned["duration_gt_24hr"] = (cleaned["duration_min"] > 24 * 60).astype("Int64")
    cleaned["duration_gt_7day"] = (cleaned["duration_min"] > 7 * 24 * 60).astype("Int64")

    summary = {
        "file_name": path.name,
        "year_inferred": infer_year(path),
        "n_rows": len(cleaned),
        "start_missing_rate": float(cleaned["start_time"].isna().mean()),
        "closed_missing_rate": float(cleaned["closed_time"].isna().mean()),
        "duration_missing_rate": float(cleaned["duration_min"].isna().mean()),
        "duration_nonpositive_rate": float((cleaned["duration_min"] <= 0).mean()),
        "duration_gt_24hr_rate": float((cleaned["duration_min"] > 24 * 60).mean()),
        "duration_gt_7day_rate": float((cleaned["duration_min"] > 7 * 24 * 60).mean()),
        "duration_median_min": float(cleaned["duration_min"].median(skipna=True)) if cleaned["duration_min"].notna().any() else np.nan,
        "duration_p90_min": float(cleaned["duration_min"].quantile(0.90)) if cleaned["duration_min"].notna().any() else np.nan,
        "duration_p99_min": float(cleaned["duration_min"].quantile(0.99)) if cleaned["duration_min"].notna().any() else np.nan,
        "n_event_types": int(cleaned["event_type"].nunique(dropna=True)),
        "n_routes_extracted": int(cleaned["route"].nunique(dropna=True)),
        "read_status": "ok",
    }

    return cleaned, summary

In [4]:
chart_frames = []
chart_quality_rows = []

for i, fp in enumerate(chart_files, start=1):
    print(f"[{i}/{len(chart_files)}] Processing {fp.name} ...")
    try:
        cleaned_df, summary_row = parse_chart_file(fp)
        chart_frames.append(cleaned_df)
        chart_quality_rows.append(summary_row)
    except Exception as e:
        chart_quality_rows.append({
            "file_name": fp.name,
            "year_inferred": infer_year(fp),
            "read_status": "error",
            "error": str(e),
        })
        print("  -> ERROR:", e)

chart_quality = pd.DataFrame(chart_quality_rows)
chart_quality.to_csv(OUTPUT_TABLE_DIR / "chart_duration_quality_summary.csv", index=False)

print("Saved:", OUTPUT_TABLE_DIR / "chart_duration_quality_summary.csv")
chart_quality

[1/5] Processing MDOT_CHART_2020.csv ...
  -> ERROR: Can only use .dt accessor with datetimelike values
[2/5] Processing MDOT_CHART_2021.csv ...
  -> ERROR: Can only use .dt accessor with datetimelike values
[3/5] Processing MDOT_CHART_2022.csv ...
  -> ERROR: unsupported operand type(s) for /: 'float' and 'Timedelta'
[4/5] Processing MDOT_CHART_2023.csv ...
  -> ERROR: unsupported operand type(s) for /: 'float' and 'Timedelta'
[5/5] Processing MDOT_CHART_2024.csv ...
  -> ERROR: unsupported operand type(s) for /: 'float' and 'Timedelta'
Saved: \\chart_duration_quality_summary.csv


,file_name,year_inferred,read_status,error
0,MDOT_CHART_2020.csv,2020,error,Can only use .dt accessor with datetimelike va...
1,MDOT_CHART_2021.csv,2021,error,Can only use .dt accessor with datetimelike va...
2,MDOT_CHART_2022.csv,2022,error,unsupported operand type(s) for /: 'float' and...
3,MDOT_CHART_2023.csv,2023,error,unsupported operand type(s) for /: 'float' and...
4,MDOT_CHART_2024.csv,2024,error,unsupported operand type(s) for /: 'float' and...


In [7]:
chart_quality

,file_name,year_inferred,read_status,error
0,MDOT_CHART_2020.csv,2020,error,Can only use .dt accessor with datetimelike va...
1,MDOT_CHART_2021.csv,2021,error,Can only use .dt accessor with datetimelike va...
2,MDOT_CHART_2022.csv,2022,error,unsupported operand type(s) for /: 'float' and...
3,MDOT_CHART_2023.csv,2023,error,unsupported operand type(s) for /: 'float' and...
4,MDOT_CHART_2024.csv,2024,error,unsupported operand type(s) for /: 'float' and...


In [8]:
chart_frames = []
chart_quality_rows = []

for i, fp in enumerate(chart_files, start=1):
    print(f"[{i}/{len(chart_files)}] Processing {fp.name} ...")

    try:
        # 먼저 헤더만 확인
        head = read_csv_safe(fp, nrows=5)
        print("  columns:", list(head.columns)[:10])

        cleaned_df, summary_row = parse_chart_file(fp)

        if cleaned_df is None or len(cleaned_df) == 0:
            chart_quality_rows.append({
                "file_name": fp.name,
                "year_inferred": infer_year(fp),
                "read_status": "empty_after_parse",
                "error": "Parsed dataframe is empty"
            })
            print("  -> parsed dataframe empty")
            continue

        chart_frames.append(cleaned_df)
        chart_quality_rows.append(summary_row)
        print(f"  -> success, rows: {len(cleaned_df):,}")

    except Exception as e:
        chart_quality_rows.append({
            "file_name": fp.name,
            "year_inferred": infer_year(fp),
            "read_status": "error",
            "error": str(e),
        })
        print("  -> ERROR:", repr(e))

chart_quality = pd.DataFrame(chart_quality_rows)
chart_quality.to_csv(OUTPUT_TABLE_DIR / "chart_duration_quality_summary.csv", index=False)

print("Saved:", OUTPUT_TABLE_DIR / "chart_duration_quality_summary.csv")
chart_quality

[1/5] Processing MDOT_CHART_2020.csv ...
  columns: ['Agency', 'Standardized Type', 'Agency-specific Type', 'Agency-specific Sub Type', 'Start time', 'Closed time', 'Location', 'Op. Center', 'Duration (Incident clearance time)', 'Operator Notes']
  -> ERROR: AttributeError('Can only use .dt accessor with datetimelike values')
[2/5] Processing MDOT_CHART_2021.csv ...
  columns: ['Agency', 'Standardized Type', 'Agency-specific Type', 'Agency-specific Sub Type', 'Start time', 'Closed time', 'Location', 'Op. Center', 'Duration (Incident clearance time)', 'Operator Notes']
  -> ERROR: AttributeError('Can only use .dt accessor with datetimelike values')
[3/5] Processing MDOT_CHART_2022.csv ...
  columns: ['Agency', 'Standardized Type', 'Agency-specific Type', 'Agency-specific Sub Type', 'Start time', 'Closed time', 'Location', 'Op. Center', 'Duration (Incident clearance time)', 'Operator Notes']
  -> ERROR: TypeError("unsupported operand type(s) for /: 'float' and 'Timedelta'")
[4/5] Process

,file_name,year_inferred,read_status,error
0,MDOT_CHART_2020.csv,2020,error,Can only use .dt accessor with datetimelike va...
1,MDOT_CHART_2021.csv,2021,error,Can only use .dt accessor with datetimelike va...
2,MDOT_CHART_2022.csv,2022,error,unsupported operand type(s) for /: 'float' and...
3,MDOT_CHART_2023.csv,2023,error,unsupported operand type(s) for /: 'float' and...
4,MDOT_CHART_2024.csv,2024,error,unsupported operand type(s) for /: 'float' and...


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import warnings

warnings.filterwarnings("ignore")

RAW_BASE = Path(r"C:")
PROJECT_ROOT = Path(r"\Traffic_Incident")

CHART_DIR = RAW_BASE / "MDOT_CHART"

OUTPUT_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_DATA_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("CHART_DIR exists:", CHART_DIR.exists())
print("OUTPUT_TABLE_DIR:", OUTPUT_TABLE_DIR)
print("OUTPUT_DATA_DIR:", OUTPUT_DATA_DIR)

CHART_DIR exists: True
OUTPUT_TABLE_DIR: \\tables
OUTPUT_DATA_DIR: \\data\processed


In [ ]:
## 3. Load Raw CHART Files

In [10]:
def list_csv_files(folder: Path):
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*.csv") if p.is_file()])


chart_files = list_csv_files(CHART_DIR)

print(f"CHART files found: {len(chart_files):,}")
for f in chart_files:
    print("-", f.name)

CHART files found: 5
- MDOT_CHART_2020.csv
- MDOT_CHART_2021.csv
- MDOT_CHART_2022.csv
- MDOT_CHART_2023.csv
- MDOT_CHART_2024.csv


In [11]:
def read_csv_safe(path, nrows=None, usecols=None):
    try:
        return pd.read_csv(path, nrows=nrows, usecols=usecols, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, nrows=nrows, usecols=usecols, encoding="latin1", low_memory=False)


def existing_col(columns, candidates):
    mapping = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        key = cand.strip().lower()
        if key in mapping:
            return mapping[key]
    return None


def infer_year(path: Path):
    m = re.search(r"(20\d{2})", str(path))
    return int(m.group(1)) if m else np.nan


def clean_string_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .str.strip()
         .replace({"<NA>": pd.NA, "nan": pd.NA, "None": pd.NA, "": pd.NA})
    )


def extract_route(location: pd.Series) -> pd.Series:
    pat = r"\b(I[-\s]?\d+|US[-\s]?\d+|MD[-\s]?\d+|SR[-\s]?\d+|RT[-\s]?\d+)\b"
    out = location.str.extract(pat, expand=False)
    return out.astype("string").str.upper().str.replace(r"\s+", "-", regex=True)


def extract_direction(location: pd.Series) -> pd.Series:
    pat = r"\b(NORTH|SOUTH|EAST|WEST|NORTHBOUND|SOUTHBOUND|EASTBOUND|WESTBOUND|INNER LOOP|OUTER LOOP|NB|SB|EB|WB)\b"
    out = location.str.extract(pat, expand=False)
    return out.astype("string").str.upper()


def extract_exit(location: pd.Series) -> pd.Series:
    pat = r"\bEXIT\s+([0-9A-Z]+)\b"
    out = location.str.extract(pat, expand=False)
    return out.astype("string").str.upper()


def extract_milepost(location: pd.Series) -> pd.Series:
    pat = r"\b(?:MP|MM|MILEPOST)\s*([0-9]+(?:\.[0-9]+)?)\b"
    out = location.str.extract(pat, expand=False)
    return pd.to_numeric(out, errors="coerce")


def parse_numeric_duration(series: pd.Series) -> pd.Series:
    """
    Duration (Incident clearance time) 컬럼을 분 단위 숫자로 변환
    숫자 문자열이면 바로 numeric
    HH:MM:SS 형태면 timedelta로 변환 후 분 계산
    """
    s = series.astype("string").str.strip()

    # 1) 숫자로 직접 변환 시도
    num = pd.to_numeric(s, errors="coerce")

    # 2) 숫자로 안 되는 값만 timedelta로 시도
    mask = num.isna() & s.notna()
    if mask.any():
        td = pd.to_timedelta(s[mask], errors="coerce")
        num.loc[mask] = td / pd.Timedelta(minutes=1)

    return num


def safe_datetime_series(series: pd.Series) -> pd.Series:
    """
    datetime dtype 강제
    """
    dt = pd.to_datetime(series, errors="coerce")
    # 확실히 datetime64[ns]로 고정
    return pd.Series(dt, index=series.index)


def safe_duration_minutes(start: pd.Series, closed: pd.Series) -> pd.Series:
    """
    start/closed 차이로 분 단위 duration 계산
    """
    start_dt = safe_datetime_series(start)
    closed_dt = safe_datetime_series(closed)

    delta = closed_dt - start_dt
    return delta.dt.total_seconds() / 60

In [ ]:
## 4. Multi-Year Data Parsing

In [15]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# PATHS
# ============================================================

RAW_BASE = Path(r"C:\")
PROJECT_ROOT = Path(r"\")

CHART_DIR = RAW_BASE / "MDOT_CHART"

OUTPUT_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_DATA_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("CHART_DIR exists:", CHART_DIR.exists())
print("OUTPUT_TABLE_DIR:", OUTPUT_TABLE_DIR)
print("OUTPUT_DATA_DIR:", OUTPUT_DATA_DIR)


# ============================================================
# BASIC HELPERS
# ============================================================

def list_csv_files(folder: Path):
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*.csv") if p.is_file()])


def read_csv_safe(path, nrows=None, usecols=None):
    try:
        return pd.read_csv(
            path,
            nrows=nrows,
            usecols=usecols,
            low_memory=False,
            encoding="utf-8-sig"
        )
    except UnicodeDecodeError:
        return pd.read_csv(
            path,
            nrows=nrows,
            usecols=usecols,
            low_memory=False,
            encoding="latin1"
        )


def existing_col(columns, candidates):
    mapping = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        key = cand.strip().lower()
        if key in mapping:
            return mapping[key]
    return None


def infer_year(path: Path):
    m = re.search(r"(20\d{2})", str(path))
    return int(m.group(1)) if m else np.nan


def clean_string_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .str.strip()
         .replace({
             "<NA>": pd.NA,
             "nan": pd.NA,
             "None": pd.NA,
             "": pd.NA
         })
    )


# ============================================================
# ROBUST DATETIME / DURATION PARSING
# ============================================================

def parse_datetime_robust(series: pd.Series) -> pd.Series:
    """
    CHART Start time / Closed time robust parser.
    Uses utc=True to avoid mixed timezone/object dtype issues.
    Returns timezone-naive datetime in America/New_York when possible.
    """
    s = series.astype("string").str.strip()
    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA
    })

    # pandas 2.x supports format="mixed"; older pandas may not.
    try:
        dt = pd.to_datetime(s, errors="coerce", utc=True, format="mixed")
    except TypeError:
        dt = pd.to_datetime(s, errors="coerce", utc=True)

    # Convert UTC to local Eastern time and remove timezone.
    # If this fails, keep the parsed datetime as-is.
    try:
        dt = dt.dt.tz_convert("America/New_York").dt.tz_localize(None)
    except Exception:
        try:
            dt = dt.dt.tz_localize(None)
        except Exception:
            pass

    return dt


def parse_duration_minutes(series: pd.Series) -> pd.Series:
    """
    Parses 'Duration (Incident clearance time)' into minutes.

    Handles:
    - numeric minutes: 29, 45.5
    - HH:MM:SS
    - days HH:MM:SS
    - '29 min', '2 hours'
    """
    raw = series.astype("string").str.strip()
    raw = raw.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA
    })

    out = pd.Series(np.nan, index=series.index, dtype="float64")

    # Case 1: pure numeric values, assume minutes
    numeric_candidate = raw.str.replace(",", "", regex=False)
    numeric_val = pd.to_numeric(numeric_candidate, errors="coerce")
    out.loc[numeric_val.notna()] = numeric_val.loc[numeric_val.notna()]

    # Case 2: values like "29 min", "29 minutes"
    minute_match = raw.str.extract(
        r"(?i)^\s*([0-9]+(?:\.[0-9]+)?)\s*(?:min|mins|minute|minutes)\s*$",
        expand=False
    )
    minute_val = pd.to_numeric(minute_match, errors="coerce")
    out.loc[minute_val.notna()] = minute_val.loc[minute_val.notna()]

    # Case 3: values like "2 hr", "2 hours"
    hour_match = raw.str.extract(
        r"(?i)^\s*([0-9]+(?:\.[0-9]+)?)\s*(?:hr|hrs|hour|hours)\s*$",
        expand=False
    )
    hour_val = pd.to_numeric(hour_match, errors="coerce") * 60
    out.loc[hour_val.notna()] = hour_val.loc[hour_val.notna()]

    # Case 4: timedelta-like strings
    remaining = out.isna() & raw.notna()
    if remaining.any():
        td = pd.to_timedelta(raw.loc[remaining], errors="coerce")
        td_min = td / pd.Timedelta(minutes=1)
        out.loc[remaining] = td_min

    return out


def calculate_duration_from_times(start_time: pd.Series, closed_time: pd.Series) -> pd.Series:
    """
    Fallback duration calculation from parsed datetimes.
    Does not use .dt accessor.
    """
    out = pd.Series(np.nan, index=start_time.index, dtype="float64")

    try:
        delta = closed_time - start_time
        out = delta / pd.Timedelta(minutes=1)
        out = pd.to_numeric(out, errors="coerce")
    except Exception:
        pass

    return out


# ============================================================
# LOCATION PARSING
# ============================================================

def extract_route(location: pd.Series) -> pd.Series:
    pat = r"\b(I[-\s]?\d+|US[-\s]?\d+|MD[-\s]?\d+|SR[-\s]?\d+|RT[-\s]?\d+)\b"
    out = location.str.extract(pat, expand=False)
    return out.astype("string").str.upper().str.replace(r"\s+", "-", regex=True)


def extract_direction(location: pd.Series) -> pd.Series:
    pat = r"\b(NORTH|SOUTH|EAST|WEST|NORTHBOUND|SOUTHBOUND|EASTBOUND|WESTBOUND|INNER LOOP|OUTER LOOP|NB|SB|EB|WB)\b"
    out = location.str.extract(pat, expand=False)
    return out.astype("string").str.upper()


def extract_exit(location: pd.Series) -> pd.Series:
    pat = r"\bEXIT\s+([0-9A-Z]+)\b"
    out = location.str.extract(pat, expand=False)
    return out.astype("string").str.upper()


def extract_milepost(location: pd.Series) -> pd.Series:
    pat = r"\b(?:MP|MM|MILEPOST)\s*([0-9]+(?:\.[0-9]+)?)\b"
    out = location.str.extract(pat, expand=False)
    return pd.to_numeric(out, errors="coerce")


# ============================================================
# MAIN PARSER
# ============================================================

def parse_chart_file(path: Path):
    df = read_csv_safe(path)
    cols = list(df.columns)

    agency_col = existing_col(cols, ["Agency"])
    type_col = existing_col(cols, ["Standardized Type"])
    agency_type_col = existing_col(cols, ["Agency-specific Type", "Agency Specific Type"])
    agency_subtype_col = existing_col(cols, ["Agency-specific Sub Type", "Agency Specific Sub Type"])
    start_col = existing_col(cols, ["Start time", "Start Time"])
    closed_col = existing_col(cols, ["Closed time", "Closed Time"])
    location_col = existing_col(cols, ["Location"])
    op_center_col = existing_col(cols, ["Op. Center", "Op Center"])
    duration_col = existing_col(cols, ["Duration (Incident clearance time)", "Duration"])
    notes_col = existing_col(cols, ["Operator Notes"])

    if duration_col is None:
        raise ValueError(f"No duration column found in {path.name}. Columns: {cols}")

    # Parse datetimes, but do not rely on them for primary duration.
    start_time = parse_datetime_robust(df[start_col]) if start_col else pd.Series(pd.NaT, index=df.index)
    closed_time = parse_datetime_robust(df[closed_col]) if closed_col else pd.Series(pd.NaT, index=df.index)

    # Primary target: CHART provided duration column
    duration_min_raw = parse_duration_minutes(df[duration_col])

    # Fallback: closed - start
    duration_min_calc = calculate_duration_from_times(start_time, closed_time)

    # Use raw duration first, then fill with calc if raw missing
    duration_min = duration_min_raw.copy()
    fill_mask = duration_min.isna() & duration_min_calc.notna()
    duration_min.loc[fill_mask] = duration_min_calc.loc[fill_mask]

    cleaned = pd.DataFrame({
        "source_file": path.name,
        "row_in_source": np.arange(len(df)),
        "event_uid": [f"{path.stem}_{i}" for i in range(len(df))],

        "year_inferred": infer_year(path),
        "agency": clean_string_series(df[agency_col]) if agency_col else pd.Series(pd.NA, index=df.index, dtype="string"),
        "event_type": clean_string_series(df[type_col]) if type_col else pd.Series(pd.NA, index=df.index, dtype="string"),
        "agency_type": clean_string_series(df[agency_type_col]) if agency_type_col else pd.Series(pd.NA, index=df.index, dtype="string"),
        "agency_subtype": clean_string_series(df[agency_subtype_col]) if agency_subtype_col else pd.Series(pd.NA, index=df.index, dtype="string"),

        "start_time": start_time,
        "closed_time": closed_time,

        "duration_min": duration_min,
        "duration_min_raw": duration_min_raw,
        "duration_min_calc": duration_min_calc,

        "location_raw": clean_string_series(df[location_col]) if location_col else pd.Series(pd.NA, index=df.index, dtype="string"),
        "op_center": clean_string_series(df[op_center_col]) if op_center_col else pd.Series(pd.NA, index=df.index, dtype="string"),
        "operator_notes": clean_string_series(df[notes_col]) if notes_col else pd.Series(pd.NA, index=df.index, dtype="string"),
    })

    # Time features.
    # These use parsed start_time. If parsing failed, they remain missing.
    if pd.api.types.is_datetime64_any_dtype(cleaned["start_time"]):
        cleaned["year"] = cleaned["start_time"].dt.year
        cleaned["month"] = cleaned["start_time"].dt.month
        cleaned["day"] = cleaned["start_time"].dt.day
        cleaned["hour"] = cleaned["start_time"].dt.hour
        cleaned["day_of_week"] = cleaned["start_time"].dt.dayofweek
        cleaned["day_name"] = cleaned["start_time"].dt.day_name()
    else:
        cleaned["year"] = np.nan
        cleaned["month"] = np.nan
        cleaned["day"] = np.nan
        cleaned["hour"] = np.nan
        cleaned["day_of_week"] = np.nan
        cleaned["day_name"] = pd.NA

    cleaned["is_weekend"] = cleaned["day_of_week"].isin([5, 6]).astype("Int64")
    cleaned["is_peak_am"] = cleaned["hour"].between(6, 9, inclusive="both").astype("Int64")
    cleaned["is_peak_pm"] = cleaned["hour"].between(15, 18, inclusive="both").astype("Int64")

    # Location features
    loc = cleaned["location_raw"].fillna("").astype("string").str.upper()
    cleaned["route"] = extract_route(loc)
    cleaned["direction"] = extract_direction(loc)
    cleaned["exit"] = extract_exit(loc)
    cleaned["milepost"] = extract_milepost(loc)

    # Quality flags
    cleaned["duration_missing"] = cleaned["duration_min"].isna().astype("Int64")
    cleaned["duration_nonpositive"] = (cleaned["duration_min"] <= 0).astype("Int64")
    cleaned["duration_gt_24hr"] = (cleaned["duration_min"] > 24 * 60).astype("Int64")
    cleaned["duration_gt_7day"] = (cleaned["duration_min"] > 7 * 24 * 60).astype("Int64")

    summary = {
        "file_name": path.name,
        "year_inferred": infer_year(path),
        "n_rows": len(cleaned),

        "start_missing_rate": float(cleaned["start_time"].isna().mean()),
        "closed_missing_rate": float(cleaned["closed_time"].isna().mean()),

        "duration_missing_rate": float(cleaned["duration_min"].isna().mean()),
        "duration_raw_missing_rate": float(cleaned["duration_min_raw"].isna().mean()),
        "duration_calc_missing_rate": float(cleaned["duration_min_calc"].isna().mean()),

        "duration_nonpositive_rate": float((cleaned["duration_min"] <= 0).mean()),
        "duration_gt_24hr_rate": float((cleaned["duration_min"] > 24 * 60).mean()),
        "duration_gt_7day_rate": float((cleaned["duration_min"] > 7 * 24 * 60).mean()),

        "duration_median_min": float(cleaned["duration_min"].median(skipna=True)) if cleaned["duration_min"].notna().any() else np.nan,
        "duration_p75_min": float(cleaned["duration_min"].quantile(0.75)) if cleaned["duration_min"].notna().any() else np.nan,
        "duration_p90_min": float(cleaned["duration_min"].quantile(0.90)) if cleaned["duration_min"].notna().any() else np.nan,
        "duration_p99_min": float(cleaned["duration_min"].quantile(0.99)) if cleaned["duration_min"].notna().any() else np.nan,

        "n_event_types": int(cleaned["event_type"].nunique(dropna=True)),
        "n_routes_extracted": int(cleaned["route"].nunique(dropna=True)),
        "read_status": "ok",
    }

    return cleaned, summary


# ============================================================
# RUN CHART PARSING
# ============================================================

chart_files = list_csv_files(CHART_DIR)

print(f"CHART files found: {len(chart_files)}")
for f in chart_files:
    print("-", f.name)

chart_frames = []
chart_quality_rows = []

for i, fp in enumerate(chart_files, start=1):
    print(f"[{i}/{len(chart_files)}] Processing {fp.name} ...")

    try:
        cleaned_df, summary_row = parse_chart_file(fp)
        chart_frames.append(cleaned_df)
        chart_quality_rows.append(summary_row)

        print(
            f"  -> success | rows={len(cleaned_df):,} | "
            f"median duration={summary_row['duration_median_min']:.2f} min"
        )

    except Exception as e:
        chart_quality_rows.append({
            "file_name": fp.name,
            "year_inferred": infer_year(fp),
            "read_status": "error",
            "error": repr(e),
        })
        print("  -> ERROR:", repr(e))

chart_quality = pd.DataFrame(chart_quality_rows)
chart_quality.to_csv(OUTPUT_TABLE_DIR / "chart_duration_quality_summary.csv", index=False)

print("\nSaved:", OUTPUT_TABLE_DIR / "chart_duration_quality_summary.csv")
display(chart_quality)


# ============================================================
# COMBINE AND SAVE CLEAN DATA
# ============================================================

if len(chart_frames) == 0:
    raise RuntimeError("No CHART files parsed successfully. Check chart_quality error column.")

chart_events_clean = pd.concat(chart_frames, ignore_index=True)

chart_events_clean.to_parquet(
    OUTPUT_DATA_DIR / "chart_events_clean.parquet",
    index=False
)

chart_events_clean.head(200).to_csv(
    OUTPUT_TABLE_DIR / "chart_events_clean_sample_head.csv",
    index=False
)

print("\nSaved:", OUTPUT_DATA_DIR / "chart_events_clean.parquet")
print("Rows:", f"{len(chart_events_clean):,}")
print("Columns:", len(chart_events_clean.columns))

display(chart_events_clean.head())


# ============================================================
# EVENT TYPE DURATION SUMMARY
# ============================================================

valid_duration = chart_events_clean.dropna(subset=["event_type", "duration_min"]).copy()

chart_event_type_duration_summary = (
    valid_duration
    .groupby("event_type", dropna=False)
    .agg(
        n_events=("duration_min", "size"),
        duration_median_min=("duration_min", "median"),
        duration_p75_min=("duration_min", lambda x: x.quantile(0.75)),
        duration_p90_min=("duration_min", lambda x: x.quantile(0.90)),
        duration_p99_min=("duration_min", lambda x: x.quantile(0.99)),
        nonpositive_rate=("duration_min", lambda x: float((x <= 0).mean())),
        gt_24hr_rate=("duration_min", lambda x: float((x > 24 * 60).mean())),
    )
    .reset_index()
    .sort_values("n_events", ascending=False)
)

chart_event_type_duration_summary.to_csv(
    OUTPUT_TABLE_DIR / "chart_event_type_duration_summary.csv",
    index=False
)

print("\nSaved:", OUTPUT_TABLE_DIR / "chart_event_type_duration_summary.csv")
display(chart_event_type_duration_summary.head(20))


# ============================================================
# MODELING-READY INCIDENT SUBSET
# ============================================================

chart_model_base = chart_events_clean.copy()

chart_model_base = chart_model_base.dropna(subset=["duration_min", "event_type"]).copy()
chart_model_base = chart_model_base[chart_model_base["duration_min"] > 0].copy()

# Long/planned/report-like event types excluded from primary incident model.
exclude_from_primary_incident_model = [
    "Road Maintenance Operations",
    "Alert",
    "Traffic Signal Not Working",
    "Special Event",
    "Winter Road Report",
]

chart_model_base["is_incident_like"] = ~chart_model_base["event_type"].isin(
    exclude_from_primary_incident_model
)

chart_model_incident = chart_model_base[
    (chart_model_base["is_incident_like"]) &
    (chart_model_base["duration_min"] <= 24 * 60)
].copy()

chart_model_incident["log_duration_min"] = np.log1p(chart_model_incident["duration_min"])
chart_model_incident["long_duration_30"] = (chart_model_incident["duration_min"] > 30).astype(int)
chart_model_incident["long_duration_60"] = (chart_model_incident["duration_min"] > 60).astype(int)
chart_model_incident["long_duration_90"] = (chart_model_incident["duration_min"] > 90).astype(int)

chart_model_incident.to_parquet(
    OUTPUT_DATA_DIR / "chart_model_incident.parquet",
    index=False
)

print("\nSaved:", OUTPUT_DATA_DIR / "chart_model_incident.parquet")
print("Model rows:", f"{len(chart_model_incident):,}")
print("Median duration:", chart_model_incident["duration_min"].median())
print("P90 duration:", chart_model_incident["duration_min"].quantile(0.90))

display(chart_model_incident.head())


# ============================================================
# FINAL CHECK
# ============================================================

print("\n=== Saved table files ===")
for p in sorted(OUTPUT_TABLE_DIR.glob("chart_*.csv")):
    print("-", p.name)

print("\n=== Saved parquet files ===")
for p in sorted(OUTPUT_DATA_DIR.glob("chart_*.parquet")):
    print("-", p.name)

CHART_DIR exists: True
OUTPUT_TABLE_DIR: \\tables
OUTPUT_DATA_DIR: \\data\processed
CHART files found: 5
- MDOT_CHART_2020.csv
- MDOT_CHART_2021.csv
- MDOT_CHART_2022.csv
- MDOT_CHART_2023.csv
- MDOT_CHART_2024.csv
[1/5] Processing MDOT_CHART_2020.csv ...
  -> success | rows=96,619 | median duration=19.00 min
[2/5] Processing MDOT_CHART_2021.csv ...
  -> success | rows=103,839 | median duration=19.00 min
[3/5] Processing MDOT_CHART_2022.csv ...
  -> success | rows=102,357 | median duration=21.00 min
[4/5] Processing MDOT_CHART_2023.csv ...
  -> success | rows=108,964 | median duration=18.00 min
[5/5] Processing MDOT_CHART_2024.csv ...
  -> success | rows=106,217 | median duration=21.00 min

Saved: \\chart_duration_quality_summary.csv


,file_name,year_inferred,n_rows,start_missing_rate,closed_missing_rate,duration_missing_rate,duration_raw_missing_rate,duration_calc_missing_rate,duration_nonpositive_rate,duration_gt_24hr_rate,duration_gt_7day_rate,duration_median_min,duration_p75_min,duration_p90_min,duration_p99_min,n_event_types,n_routes_extracted,read_status
0,MDOT_CHART_2020.csv,2020,96619,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.110237,0.083131,19.0,89.0,2724.4,132742.32,20,498,ok
1,MDOT_CHART_2021.csv,2021,103839,0.0,0.000000,0.000000,0.000010,0.000000,0.000010,0.098181,0.072535,19.0,80.0,1384.6,143351.28,20,502,ok
2,MDOT_CHART_2022.csv,2022,102357,0.0,0.000029,0.000029,0.000029,0.000029,0.000000,0.102465,0.077826,21.0,84.0,1698.7,151615.69,20,496,ok
3,MDOT_CHART_2023.csv,2023,108964,0.0,0.000009,0.000009,0.000009,0.000009,0.000000,0.103722,0.080293,18.0,71.0,1780.8,153033.96,20,512,ok
4,MDOT_CHART_2024.csv,2024,106217,0.0,0.000132,0.000132,0.000141,0.000132,0.000009,0.107544,0.083235,21.0,84.0,2157.4,180366.12,20,518,ok



Saved: \\\chart_events_clean.parquet
Rows: 517,996
Columns: 33


,source_file,row_in_source,event_uid,year_inferred,agency,event_type,agency_type,agency_subtype,start_time,closed_time,...,is_peak_am,is_peak_pm,route,direction,exit,milepost,duration_missing,duration_nonpositive,duration_gt_24hr,duration_gt_7day
0,MDOT_CHART_2020.csv,0,MDOT_CHART_2020_0,2020,MDDOT,Disabled Vehicle,Disabled Vehicle,<NA>,2020-05-13 18:13:00,2020-05-13 18:46:00,...,0,1,I-695,OUTER LOOP,8B,<NA>,0,0,0,0
1,MDOT_CHART_2020.csv,1,MDOT_CHART_2020_1,2020,MDDOT,Disabled Vehicle,Disabled Vehicle,<NA>,2020-05-27 16:18:00,2020-05-27 16:19:00,...,0,1,I-95,NORTH,67,<NA>,0,0,0,0
2,MDOT_CHART_2020.csv,2,MDOT_CHART_2020_2,2020,MDDOT,Incident,Incident,<NA>,2020-05-17 10:50:00,2020-05-17 12:39:00,...,0,0,MD-295,NORTH,<NA>,<NA>,0,0,0,0
3,MDOT_CHART_2020.csv,3,MDOT_CHART_2020_3,2020,MDDOT,Disabled Vehicle,Disabled Vehicle,<NA>,2020-05-01 08:06:00,2020-05-01 08:07:00,...,1,0,I-695,OUTER LOOP,6A,<NA>,0,0,0,0
4,MDOT_CHART_2020.csv,4,MDOT_CHART_2020_4,2020,MDDOT,Road Maintenance Operations,Planned Roadway Closure,<NA>,2020-05-21 23:52:00,2020-05-22 03:01:00,...,0,0,I-95,NORTH,74,71.3,0,0,0,0



Saved: \\\chart_event_type_duration_summary.csv


,event_type,n_events,duration_median_min,duration_p75_min,duration_p90_min,duration_p99_min,nonpositive_rate,gt_24hr_rate
4,Disabled Vehicle,226352,6.0,19.00,39.0,103.00,0.000004,0.000004
2,Collision,62974,29.0,53.00,89.0,295.00,0.000000,0.000127
11,Road Maintenance Operations,62292,23381.5,59513.50,131302.4,486908.71,0.000016,0.778318
10,Obstructions,50556,13.0,32.00,92.0,394.00,0.000000,0.001484
0,Alert,28551,107.0,420.00,1430.0,24211.00,0.000000,0.081013
7,Incident,25397,25.0,132.00,378.0,1441.04,0.000000,0.010119
8,Injuries Involved,19135,50.0,78.00,126.0,335.00,0.000000,0.000157
15,Traffic Signal Not Working,15582,150.0,401.75,3237.0,28975.04,0.000000,0.156013
5,Emergency Roadwork,6539,145.0,298.00,437.0,1323.96,0.000000,0.008717
1,Animal Struck,6059,2.0,5.00,48.0,888.42,0.000000,0.002806



Saved: \\chart_model_incident.parquet
Model rows: 408,904
Median duration: 13.0
P90 duration: 77.0


,source_file,row_in_source,event_uid,year_inferred,agency,event_type,agency_type,agency_subtype,start_time,closed_time,...,milepost,duration_missing,duration_nonpositive,duration_gt_24hr,duration_gt_7day,is_incident_like,log_duration_min,long_duration_30,long_duration_60,long_duration_90
0,MDOT_CHART_2020.csv,0,MDOT_CHART_2020_0,2020,MDDOT,Disabled Vehicle,Disabled Vehicle,<NA>,2020-05-13 18:13:00,2020-05-13 18:46:00,...,<NA>,0,0,0,0,True,3.526361,1,0,0
1,MDOT_CHART_2020.csv,1,MDOT_CHART_2020_1,2020,MDDOT,Disabled Vehicle,Disabled Vehicle,<NA>,2020-05-27 16:18:00,2020-05-27 16:19:00,...,<NA>,0,0,0,0,True,0.693147,0,0,0
2,MDOT_CHART_2020.csv,2,MDOT_CHART_2020_2,2020,MDDOT,Incident,Incident,<NA>,2020-05-17 10:50:00,2020-05-17 12:39:00,...,<NA>,0,0,0,0,True,4.691348,1,1,1
3,MDOT_CHART_2020.csv,3,MDOT_CHART_2020_3,2020,MDDOT,Disabled Vehicle,Disabled Vehicle,<NA>,2020-05-01 08:06:00,2020-05-01 08:07:00,...,<NA>,0,0,0,0,True,0.693147,0,0,0
6,MDOT_CHART_2020.csv,6,MDOT_CHART_2020_6,2020,MDDOT,Disabled Vehicle,Disabled Vehicle,<NA>,2020-05-22 18:14:00,2020-05-22 18:18:00,...,<NA>,0,0,0,0,True,1.386294,0,0,0



=== Saved table files ===
- chart_duration_quality_summary.csv
- chart_event_type_duration_summary.csv
- chart_events_clean_sample_head.csv

=== Saved parquet files ===
- chart_events_clean.parquet
- chart_model_incident.parquet
